# GOAT-Evolution — Unsloth fine-tune (Colab)

Same per-attack pipeline as the TRL notebook but uses [Unsloth](https://github.com/unslothai/unsloth) for **2-3x faster** LoRA fine-tuning on free Colab T4 GPUs.

Steps:
1. Install Unsloth + TRL + the GOAT-Evolution repo.
2. Generate per-attack JSONL shards by playing episodes against the env (Gemini Red/Blue/Judge).
3. Load Qwen2.5-3B with Unsloth in 4-bit, attach LoRA.
4. Fine-tune with TRL `SFTTrainer`.
5. Plot loss + before/after holdout.
6. Push the adapter to Hugging Face.

## 1. Install Unsloth + deps

In [ ]:
%pip install -q -U \
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" \
    "trl>=0.13" \
    "peft>=0.12" \
    "datasets>=2.20" \
    "accelerate>=0.34" \
    "huggingface_hub>=0.26" \
    "openenv-core>=0.2.1" \
    "google-generativeai>=0.8.3" \
    "matplotlib>=3.8" \
    "pydantic>=2.9" \
    "fastapi>=0.115" \
    "sse-starlette>=2.1" \
    "httpx>=0.27"

## 2. Clone the repo

In [ ]:
import os, sys, subprocess, pathlib
REPO_URL = os.environ.get('GOAT_REPO_URL', 'https://github.com/YOUR_USER/goat-evolution.git')
REPO_DIR = pathlib.Path('/content/goat-evolution')
if not REPO_DIR.exists():
    subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)], check=False)
PROJECT = REPO_DIR / 'goat_evolution' if (REPO_DIR / 'goat_evolution').exists() else REPO_DIR
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)
print('cwd:', os.getcwd())

## 3. Secrets

In [ ]:
import os
try:
    from google.colab import userdata
    for key in ('HF_TOKEN', 'GEMINI_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
except Exception:
    pass
for key in ('HF_TOKEN', 'GEMINI_API_KEY'):
    if not os.environ.get(key):
        os.environ[key] = input(f'Paste {key}: ').strip()
from huggingface_hub import login
login(os.environ['HF_TOKEN'])

## 4. Configuration

In [ ]:
MODEL_BASE      = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MODEL_OUT       = 'YOUR_USER/goat-evolution-3b-defender-unsloth'
ARTIFACTS_DIR   = '/content/artifacts'
OUT_DIR         = '/content/unsloth_out'
EPISODES_TO_GEN = 5
EPOCHS          = 1
BATCH_SIZE      = 2
GRAD_ACCUM      = 4
LR              = 2e-4
MAX_LEN         = 1024
LORA_R          = 16
PUSH            = False
ONLY_BREAKING   = True

## 5. Generate per-attack shards from the env

In [ ]:
import asyncio, pathlib
from backend.cli.realtime_trainer import RealtimeTrainer
from backend.training.llm_factory import build_agents
from backend.training.logging_setup import setup_logging
setup_logging(logs_root='/content/logs', run_label='unsloth')
agents = build_agents(
    red_provider_name='gemini',
    blue_provider_name='gemini',
    judge_provider_name='gemini',
)
trainer_play = RealtimeTrainer(
    artifacts_root=pathlib.Path(ARTIFACTS_DIR),
    subject_size='3b',
    backend='stub',  # data-gen only; real training happens with Unsloth below
    push_each_attack=False,
    hf_dataset_repo=None,
    hf_model_repo=None,
    gate_min_violation_drop=0.0,
    gate_max_refusal_drop=0.05,
    learning_rate=5e-5,
    agents=agents,
)
async def _gen():
    for i in range(EPISODES_TO_GEN):
        r = await trainer_play.run_episode(task_id=4, model_profile='standard_model')
        print(f'  ep {i+1}: turns={r.turns} breaking={r.breaking_attacks}')
asyncio.run(_gen())
DATASET_JSONL = pathlib.Path(ARTIFACTS_DIR) / 'dataset' / 'realtime_samples.jsonl'
print('Dataset rows file:', DATASET_JSONL, '->', DATASET_JSONL.stat().st_size, 'bytes')

## 6. Load Unsloth model (4-bit Qwen2.5-3B)

In [ ]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_BASE,
    max_seq_length=MAX_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_R,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

## 7. Load + format the per-attack dataset

In [ ]:
import json
from datasets import Dataset
rows = []
with open(DATASET_JSONL) as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        if ONLY_BREAKING and not rec.get('is_breaking'):
            continue
        rows.append(rec)
print('Training rows:', len(rows))
def to_text(row):
    prompt = row.get('prompt', '')
    chosen = row.get('chosen_response', '')
    return {'text': f'<|user|>\n{prompt}\n<|assistant|>\n{chosen}'}
dataset = Dataset.from_list([to_text(r) for r in rows])
dataset

## 8. Train with TRL SFTTrainer (Unsloth-accelerated)

In [ ]:
from trl import SFTConfig, SFTTrainer
args = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    max_length=MAX_LEN,
    logging_steps=2,
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    report_to=[],
    push_to_hub=PUSH,
    hub_model_id=MODEL_OUT if PUSH else None,
)
trainer_sft = SFTTrainer(model=model, args=args, train_dataset=dataset, processing_class=tokenizer)
result = trainer_sft.train()
trainer_sft.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print('Train metrics:', result.metrics)

## 9. Loss curve

In [ ]:
import matplotlib.pyplot as plt
history = trainer_sft.state.log_history
steps  = [h['step'] for h in history if 'loss' in h]
losses = [h['loss'] for h in history if 'loss' in h]
plt.figure(figsize=(8,4))
plt.plot(steps, losses, label='train loss (Unsloth)')
plt.xlabel('training step'); plt.ylabel('loss'); plt.title('Unsloth SFT loss curve')
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/loss_curve.png', dpi=130); plt.show()

## 10. Baseline vs trained on the GOAT-Evolution holdout

In [ ]:
import pathlib
from backend.training.gating import HoldoutGate
from backend.training.llm_subject import LLMSubject
FastLanguageModel.for_inference(trainer_sft.model)
subject_trained = LLMSubject(backend='hf_local', model_name=MODEL_BASE, adapter_dir=pathlib.Path(OUT_DIR))
subject_trained._tokenizer = tokenizer
subject_trained._model = trainer_sft.model
subject_trained._hf_loaded = True
subject_baseline = LLMSubject(backend='stub', model_name=MODEL_BASE)
gate = HoldoutGate()
before = gate.evaluate(subject_baseline)
after  = gate.evaluate(subject_trained)
print('Before:', before.to_dict()); print('After :', after.to_dict())

In [ ]:
import matplotlib.pyplot as plt
labels = ['asr', 'avg_violation', 'refusal_quality']
b = [before.asr, before.avg_violation, before.refusal_quality]
a = [after.asr,  after.avg_violation,  after.refusal_quality]
x = range(len(labels))
plt.figure(figsize=(7,4))
plt.bar([i-0.18 for i in x], b, width=0.36, label='baseline')
plt.bar([i+0.18 for i in x], a, width=0.36, label='Unsloth-trained')
plt.xticks(list(x), labels); plt.ylabel('score'); plt.title('Holdout: baseline vs Unsloth-trained')
plt.legend(); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/holdout_compare.png', dpi=130); plt.show()

## 11. Push the LoRA adapter (optional)

In [ ]:
if PUSH:
    trainer_sft.model.push_to_hub(MODEL_OUT, save_method='lora')
    tokenizer.push_to_hub(MODEL_OUT)
    print('Pushed:', f'https://huggingface.co/{MODEL_OUT}')